# LocalExploreAI — DATA_ngay17 offline experiments

This notebook trains an exploratory Ranker and Flood Risk baseline from `data/dataset/`. **The supplied generator creates simulated clicks/bookings, formula-derived flood labels, placeholder evidence URLs, and fabricated verification metadata. These models are not production eligible and must not be used for travel or flood decisions.**

The Ranker uses the API's versioned feature builder so the feature names can be checked. Its metadata disables deployment and pilot runtime. The Flood artifact is marked `demo_mode=true`; the API's loader must refuse it.

## 1. Install the optional training dependencies

Run this from the repository root. If dependencies are already installed in your project environment, skip the next cell.

In [ ]:
%pip install -r apps/api/requirements-dataset17-training.txt

## 2. Train and export the experiment bundles

The script checks that the supplied generator contains its known simulation markers before training. It writes `.pkl`/`.joblib` artifacts, metrics, schema files, metadata, and a model card under `models/dataset-17-experiment/`, with separate `ranker/` and `flood-risk/` bundles. Re-running updates the named outputs in those folders.

In [ ]:
%run scripts/train_dataset_17_experiments.py --dataset-dir data/dataset --output-dir models/dataset-17-experiment

## 3. Review the metrics and deployment flags

Metrics here quantify how well the models imitate the synthetic generator. A high score is not evidence that the model predicts real traveler choices or actual flooding.

In [ ]:
import json
from pathlib import Path

out = Path('models/dataset-17-experiment')
report = json.loads((out / 'experiment_report.json').read_text(encoding='utf-8'))
print('Ranker NDCG@5:', report['ranker']['test_ndcg_at_5'])
print('Flood test metrics:', report['flood']['test'])
print('Ranker production eligible:', json.loads((out / 'ranker/model_metadata.json').read_text())['production_eligible'])
print('Flood demo mode:', json.loads((out / 'flood-risk/metadata.json').read_text())['demo_mode'])

## 4. Check that the application blocks the artifacts

Run this cell in the repository environment with API dependencies installed. It verifies the Ranker falls back to the rule baseline and the Flood bundle remains blocked.

In [ ]:
!python scripts/check_dataset_17_artifacts.py --artifact-dir models/dataset-17-experiment

## What is required before production

For Ranker, collect consented impressions and genuine click/save/booking/completion/cancellation outcomes, with viewability and outcome windows, then evaluate on time-separated requests and users. For Flood Risk, replace the simulated status/evidence with audited source-backed observations, include sufficiently covered non-flood windows, align data to the production API's feature contract, and validate on held-out places and future time periods. Do not promote these bundles by editing metadata flags.